# Lab: Classification with Bank Marketing Data

---

In this lab you will practice building and evaluating **classification models** using a real bank marketing dataset.

The data comes from direct marketing campaigns conducted by a Portuguese banking institution. The campaigns were based on phone calls, and the classification goal is to predict whether a client will subscribe to a term deposit.

We will use the official `bank.csv` sample from the **UCI Bank Marketing dataset**. It is a 10% random sample of the larger dataset and is useful here because we will train several KNN models while choosing a value of `k`.

**Source:** UCI Machine Learning Repository — Bank Marketing  
https://archive.ics.uci.edu/dataset/222/bank+marketing

**Target:** `y`
- `yes` = client subscribed to a term deposit
- `no` = client did not subscribe

### Data Dictionary

| Variable | Description |
|---|---|
| `age` | Client age |
| `job` | Type of job |
| `marital` | Marital status |
| `education` | Education level |
| `default` | Whether the client has credit in default |
| `balance` | Average yearly account balance, in euros |
| `housing` | Whether the client has a housing loan |
| `loan` | Whether the client has a personal loan |
| `contact` | Contact communication type |
| `day` | Day of the month of the last contact |
| `month` | Month of the last contact |
| `duration` | Duration of the last contact, in seconds |
| `campaign` | Number of contacts performed during the current campaign |
| `pdays` | Days since the client was last contacted in a previous campaign (`-1` means not previously contacted) |
| `previous` | Number of contacts performed before the current campaign |
| `poutcome` | Outcome of the previous marketing campaign |
| `y` | Whether the client subscribed to the term deposit (`yes` / `no`) |

#### Understanding the Bank Marketing Features

Think of each row as describing a client at the time of a contact in the **current marketing campaign**.

##### Timeline

```text
CLIENT INFORMATION — already known
│
├── age        → client's age
├── job        → occupation
├── marital    → marital status
├── education  → education level
├── default    → has credit in default?
├── balance    → account balance
├── housing    → has housing loan?
└── loan       → has personal loan?
│
│
▼
PREVIOUS MARKETING CAMPAIGN(S)
│
├── previous   → number of contacts made BEFORE the current campaign
├── poutcome   → outcome of the previous campaign
│
└── Last contact from a previous campaign
        │
        │
        │  pdays
        │  = number of days since this previous contact
        │
        ▼
CURRENT MARKETING CAMPAIGN
│
├── Current contact is about to take place
│
├── month      → month of the current contact
├── day        → day of the current contact
├── contact    → contact method (e.g. cellular / telephone)
├── campaign   → contact number during this campaign (includes the current contact)
│
│                 Example:
│                 Call 1 → Call 2 → Call 3
│                                      ↑
│                                campaign = 3
│
▼
CURRENT PHONE CALL
│
├── duration   → how long this current contact/call lasted
│
▼
CAMPAIGN OUTCOME
│
└── y          → Did the client subscribe to a term deposit?
    
```

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler

### 1. Download and read in the Bank Marketing data.

Download the UCI Bank Marketing archive and extract `bank.csv`.

Then:

1. Read the CSV into a pandas dataframe.
2. Remember that this file uses `;` as the separator.
3. Display the first five rows.

In [31]:
df = pd.read_csv('bank.csv', sep=';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


### 2. Conduct an initial exploration of the dataset.

Complete the following:

1. Check the shape of the dataset.
2. Inspect the data types and missing values.
3. Display summary statistics.
4. Check the distribution of the target variable `y`, including percentages.
5. Is this a balanced or imbalanced classification problem? Explain briefly.

In [32]:
df.shape

(4521, 17)

In [33]:
df['default'].unique()

<StringArray>
['no', 'yes']
Length: 2, dtype: str

In [34]:
df.info()
# marital column has 3 unique values, i recommend one-hot encoding
# education column has 4 unique values, idk want to recommend since there is a value called 'unknown'
# [marital (3), default, housing, loan, contact(3)] columns, one-hot encoding (متعثر في السداد)
# [month] columns, ordinal or label encoding 

<class 'pandas.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        4521 non-null   int64
 1   job        4521 non-null   str  
 2   marital    4521 non-null   str  
 3   education  4521 non-null   str  
 4   default    4521 non-null   str  
 5   balance    4521 non-null   int64
 6   housing    4521 non-null   str  
 7   loan       4521 non-null   str  
 8   contact    4521 non-null   str  
 9   day        4521 non-null   int64
 10  month      4521 non-null   str  
 11  duration   4521 non-null   int64
 12  campaign   4521 non-null   int64
 13  pdays      4521 non-null   int64
 14  previous   4521 non-null   int64
 15  poutcome   4521 non-null   str  
 16  y          4521 non-null   str  
dtypes: int64(7), str(10)
memory usage: 600.6 KB


In [35]:
df.describe(include='number')

,age,balance,day,duration,campaign,pdays,previous
count,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000,4521.000000
mean,41.170095,1422.657819,15.915284,263.961292,2.793630,39.766645,0.542579
std,10.576211,3009.638142,8.247667,259.856633,3.109807,100.121124,1.693562
min,19.000000,-3313.000000,1.000000,4.000000,1.000000,-1.000000,0.000000
25%,33.000000,69.000000,9.000000,104.000000,1.000000,-1.000000,0.000000
50%,39.000000,444.000000,16.000000,185.000000,2.000000,-1.000000,0.000000
75%,49.000000,1480.000000,21.000000,329.000000,3.000000,-1.000000,0.000000
max,87.000000,71188.000000,31.000000,3025.000000,50.000000,871.000000,25.000000


In [36]:
df.describe(include= 'object')

C:\Users\TALAL ALOTHMAN\AppData\Local\Temp\ipykernel_9648\3793228563.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include= 'object')


,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521
unique,12,3,4,2,2,2,3,12,4,2
top,management,married,secondary,no,yes,no,cellular,may,unknown,no
freq,969,2797,2306,4445,2559,3830,2896,1398,3705,4000


In [37]:
print(df['y'].value_counts())
print(df['y'].value_counts(normalize=True))

y
no     4000
yes     521
Name: count, dtype: int64
y
no     0.88476
yes    0.11524
Name: proportion, dtype: float64


In [38]:
# Q5)
# imbalance


### 3. Define the business problem and choose an evaluation metric.

Assume the bank wants to use the model to decide **which clients should be prioritized for future marketing calls**.

Answer the following:

1. What does a **False Positive** mean in this business problem? What is the cost of this error?
2. What does a **False Negative** mean? What is the cost of this error?
3. Assume the bank considers both errors important, which metric would you choose?
4. Why might **Accuracy** be misleading for this dataset?

#### Answers

1. it means that our model made a false prediction that the person will subscribe to the term deposit. in this case, we annoyed the client & waste time.  
2. it means that our model made a false prediction that the person will not subscribe to the term deposit. in this case, we'll lose a deal. 
3. 1-recall (% of false negative), 1-percision (% of false positive). F1 combine both metrics in one metric
4. accuracy is misleading since it just shows the % of true predictions u made, it isn't measuring performance of  

### 4. Check for information that would lead to data leakage.

The goal is to prioritize clients **before making the marketing call**.

Review the data dictionary and identify any variable that would only become known after the call has already taken place.

1. Which variable should be removed?
2. Why would keeping it create a data leakage / unrealistic prediction problem?
3. Drop the variable from the dataframe.

#### Answers:

1. [duration, y]
2. 

??

??

In [39]:
# 3. 
df = df.drop(columns='duration')
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,jul,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,may,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,aug,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,feb,4,211,3,other,no


### 5. Encode binary categorical variables

Complete the following:

1. Identify the predictor columns containing only `yes` and `no`.
2. Map these columns to `0` and `1`.

> This mapping can be done before the train/test split because it follows a fixed rule and does not learn any information from the dataset.

In [40]:
#df[['default', 'housing', 'loan', 'y']].replace(to_replace=['yes', 'not'], value=[1, 0], inplace=True)     # why it didn't work!?
df[['default', 'housing', 'loan', 'y']] = df[['default', 'housing', 'loan', 'y']].replace({'yes':1, 'no':0}, inplace=True)
df

C:\Users\TALAL ALOTHMAN\AppData\Local\Temp\ipykernel_9648\1677997422.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[['default', 'housing', 'loan', 'y']] = df[['default', 'housing', 'loan', 'y']].replace({'yes':1, 'no':0}, inplace=True)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,oct,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,may,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,apr,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,jun,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,may,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,0,-333,1,0,cellular,30,jul,5,-1,0,unknown,0
4517,57,self-employed,married,tertiary,1,-3313,1,1,unknown,9,may,1,-1,0,unknown,0
4518,57,technician,married,secondary,0,295,0,0,cellular,19,aug,11,-1,0,unknown,0
4519,28,blue-collar,married,secondary,0,1137,0,0,cellular,6,feb,4,211,3,other,0


### 6. Prepare the target and split the data.

Complete the following:

1. Create `X` and `y`.
2. Split the data into training and testing sets.

In [41]:
X = df[['loan', 'housing', 'default', 'marital', 'job', 'poutcome', 'balance', 'contact', 'campaign', 'pdays', 'previous']]
y = df['y']

### 7. Preprocess the training and testing sets

The dataset contains both numerical and categorical variables. Also, there is imbalance in your dataset

#### Part A: Scaling the numerical features

For now, let's transform the numerical ones.

Complete the following:

1. Create a list of numerical columns.
2. Create a `StandardScaler` for the numerical columns.
    - Fit the scaler on the numerical columns of `X_train` only.
    - Use the fitted scaler to transform both `X_train` and `X_test`.
3. Store the scaled numerical values back into their original columns in `X_train` and `X_test`
    - No need to use `.set_output()`
4. Why do we need to scale our data if our model of choice is KNN?


In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=34, stratify=y)

In [43]:
num_cols = X.select_dtypes(include= 'number').columns.to_list()
print(num_cols)


['balance', 'campaign', 'pdays', 'previous']


In [44]:
Sscaler = StandardScaler().set_output(transform='pandas')

X_train_num_scaled = Sscaler.fit_transform(X_train[num_cols])
X_train_num_scaled.head()

X_test_num_scaled = Sscaler.transform(X_test[num_cols])
X_test_num_scaled.head()

,balance,campaign,pdays,previous
934,-0.111998,-0.261447,-0.403029,-0.320713
3676,0.301229,-0.588376,-0.403029,-0.320713
189,-0.865063,-0.261447,-0.403029,-0.320713
2262,-0.433654,-0.588376,-0.403029,-0.320713
2729,-0.132163,-0.588376,-0.403029,-0.320713


In [45]:
# 3. save them into original X_train & X_test
X_train[num_cols] = X_train_num_scaled
X_train.head()

X_test[num_cols] = X_test_num_scaled
X_test.head()

,loan,housing,default,marital,job,poutcome,balance,contact,campaign,pdays,previous
934,0,1,0,married,management,unknown,-0.111998,unknown,-0.261447,-0.403029,-0.320713
3676,0,0,0,married,services,unknown,0.301229,cellular,-0.588376,-0.403029,-0.320713
189,0,1,0,married,retired,unknown,-0.865063,cellular,-0.261447,-0.403029,-0.320713
2262,0,0,0,single,management,unknown,-0.433654,cellular,-0.588376,-0.403029,-0.320713
2729,0,1,0,single,blue-collar,unknown,-0.132163,unknown,-0.588376,-0.403029,-0.320713


**Q4)**

??
idk
??

??

#### Part B: Balancing the training data

Regular `SMOTE` creates synthetic observations by interpolating between neighboring observations. This works well for numerical features, but it is not suitable for categorical features.

Instead, we will use **SMOTENC**, a version of SMOTE designed for datasets containing both numerical and categorical features.

We will apply SMOTENC **before One-Hot Encoding**.

**Why?**

- After One-Hot Encoding, one categorical feature such as `job` becomes several binary columns:

    - `job_admin`, `job_student`, `job_retired`, ...
    
- If we used regular SMOTE after One-Hot Encoding, interpolation could create invalid values such as:

    - `job_admin = 0.4`, `job_student = 0.6`

- which do not represent a real category.

- So applying the resampler at this stage allows us to assign the synthetic sample the most common category among the selected minority-class neighbors.

Complete the following:

1. Identify the categorical columns in the training set.
2. Create a `SMOTENC` resampler:
   - specify which columns are categorical
   - use the default value for `k_neighbors`
   - let the minority class size be **50% of the majority class**
3. Apply the `SMOTENC` **only to the training set**.
4. Compare the training target distribution before and after `SMOTENC`.
5. Why should `SMOTENC` only be applied to the training data?
6. What is the effect of `k_neighbors` when `SMOTENC` creates synthetic observations?

**Q5)**

??

??

??

**Q6)**

??

??

??

#### Part C: Encoding the categorical features

After resampling our trainig set, we can start encoding.

Complete the following:

1. Create a `OneHotEncoder` for the categorical columns.
    - Use `.set_output()` for this part
    - Fit the encoder on the categorical columns of your **balanced training** set.
    - Use the fitted encoder to transform both the **balanced training** and **unbalanced testing** set.

2. Concatenate the numerical and encoded categorical features to create the final processed training and testing datasets.
    - For the training set, extract the numerical features from the **balanced transformed** training data and combine them with the **encoded categorical** training features.
    - For the test set, extract the numerical features from the **unbalanced transformed** test data and combine them with the **encoded categorical** test features.


In [46]:
df['poutcome'].unique()

<StringArray>
['unknown', 'failure', 'other', 'success']
Length: 4, dtype: str

In [47]:
# -------------------------
# Encode categorical features
# -------------------------
# One-hot encode the neighborhood variable

# Step 1. Instantiate the transformer
ohe = OneHotEncoder(
                drop="first", #drops first column
                sparse_output=False,  #to not compress the results
                    ).set_output(transform = "pandas")

# Step 2 & 3. Fit and transform
X_train_cat_enc = ohe.fit_transform(X_train[['marital', 'contact', 'poutcome']])

# add the encoded columns to the train data
X_train.drop(columns=['marital', 'contact', 'poutcome'], inplace=True)      # 1. droping the original columns
print(X_train.columns.to_list())

X_train[X_train_cat_enc.columns.to_list()] = X_train_cat_enc    # 2. adding the encoded columns
X_train

# during fit, the encoder learns the different categories in features
# during transform, the data is labelled 

['loan', 'housing', 'default', 'job', 'balance', 'campaign', 'pdays', 'previous']


,loan,housing,default,job,balance,campaign,pdays,previous,marital_married,marital_single,contact_telephone,contact_unknown,poutcome_other,poutcome_success,poutcome_unknown
2481,0,0,0,management,-0.158610,-0.588376,-0.403029,-0.320713,1.0,0.0,0.0,0.0,0.0,0.0,1.0
570,0,1,0,entrepreneur,-0.386381,-0.261447,-0.403029,-0.320713,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2924,0,1,0,blue-collar,0.207675,-0.588376,1.263784,0.911960,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1368,0,0,0,self-employed,-0.282578,-0.261447,-0.403029,-0.320713,0.0,1.0,0.0,1.0,0.0,0.0,1.0
3417,0,1,0,services,0.189162,-0.588376,-0.403029,-0.320713,0.0,0.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1640,0,1,0,management,-0.462414,-0.588376,-0.403029,-0.320713,1.0,0.0,0.0,1.0,0.0,0.0,1.0
71,0,1,1,entrepreneur,-0.465720,-0.588376,-0.403029,-0.320713,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1395,0,0,0,technician,0.602720,-0.261447,-0.403029,-0.320713,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2409,0,0,0,admin.,0.078748,-0.261447,-0.403029,-0.320713,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [48]:
# -------------------------
# Combine numerical + encoded categorical features
# -------------------------



---

## Logistic Regression

### 8. Train a Logistic Regression classification model.

Now:

1. Create a `LogisticRegression` model.
2. Fit the model 

### 9. Evaluate the Logistic Regression model.

Generate predictions and predicted probabilities for the test set.

Calculate:

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC

Also:

1. Display the confusion matrix.
2. Plot the ROC curve. 
    - How would you interpret the graph?
3. Based on the business objective from Question 3, which metric should receive the most attention?

??

??

??

### 10. Interpret the Logistic Regression coefficients.

1. Extract the fitted Logistic Regression coefficients and convert them into **odds ratios** using:

```python
np.exp(coefficient)
```

2. Create a dataframe containing:

- feature name
- coefficient
- odds ratio

Then:

3. Sort the table by coefficient.
4. Identify features associated with higher odds of subscription.
5. Identify features associated with lower odds of subscription.
6. Interpret at least **two odds ratios** in words.

**Important:**
- Numerical variables were standardized, so a one-unit change for them means approximately a **one standard deviation increase**.
- One-hot encoded categorical variables are interpreted relative to the category that was dropped as the reference group.

??

??

??

---

## K-Nearest Neighbors

### 11. Train a KNN classification model.

#### Part A: Train KNN models for different values of `k`

Run a KNN model for several odd values of `k` and capture the resulting metric you previously selected in a list.

Every loop iteration should:

1. Train a model with a specific `k` value.
2. Calculate the training score.
3. Calculate the testing score.
4. Create a dictionary storing `k`, the training score and the testing score.
5. Append the dictionary to a `scores` list.

You can test values such as:

```python
range(3, 22, 2)
```

#### Part B: Plot the results for the different values of `k`

1. In the same plot, graph the training and testing scores against `k`.
2. Choose a value of `k` that gives strong performance without a large gap between training and test values.

#### Part C:  Build the final KNN model.

Using the `k` value selected in Question 11:

1. Train your final KNN model

### 12. Evaluate the final KNN model.

Using the untouched test set, calculate the same metrics used for Logistic Regression:

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC

Also display:

1. The confusion matrix.
2. The ROC curve.

### 14. Compare the models and make a business recommendation.

Create a table comparing Logistic Regression and KNN using the test-set metrics.

Then answer:

1. Which model performs better for the business objective defined earlier?
2. What trade-off do you see between Precision and Recall?
3. Would you recommend Logistic Regression or KNN to the marketing team? Why?
4. Besides predictive performance, what advantage does Logistic Regression have when the business wants to understand **why** a client is more or less likely to subscribe?

Your recommendation should be based on the business objective — not simply on whichever model has the highest Accuracy.

??

??

??